# Transformer Sentiment Analysis

## 1. Setup & Configs

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no CUDA")
print(torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else "")

False
no CUDA



*Compute Environment*

No Cuda device on this local machine. Training runs on CPU.

Fine-tuning will use a subsample of the training data.

The test set stays the full split. 

In [33]:
import os
import joblib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, average_precision_score

import tqdm



In [10]:
MODEL_CKPT = 'distilbert-base-uncased'
RANDOM_STATE = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MAX_LENGTH = None
N_TRAIN = None
N_SPLITS = 5

# Paths
DATA_PATH    = "../data/processed_data/reviews_clean_v2.parquet"
BASELINE_DIR = "../models/sentiment"              # Notebook 04 artifacts
MODEL_DIR    = "../models/transformer_sentiment"  # this notebook

os.makedirs(MODEL_DIR, exist_ok=True)

# CPU threading
torch.set_num_threads(os.cpu_count())

## 2. Load Data & restore split

### 2.1 Load reviews

In [4]:
# old = pd.read_parquet('../data/processed_data/reviews_clean.parquet')
# new = pd.read_parquet('../data/processed_data/reviews_clean_v2.parquet')

# print(new['review_clean'].equals(old['review_clean']))

In [5]:
data = pd.read_parquet(DATA_PATH)

print(data.shape)

(406781, 11)


In [6]:
data.head()

,funny,helpful,hour_played,is_early_access_review,review,title,label,review_clean,language,word_count,is_duplicate
0,2.0,4,578,False,&gt Played as German Reich&gt Declare war on B...,Expansion - Hearts of Iron IV: Man the Guns,1,> played as german reich> declare war on belgi...,EN,31,False
1,0.0,0,184,False,yes.,Expansion - Hearts of Iron IV: Man the Guns,1,yes.,EN,1,False
2,0.0,0,892,False,Very good game although a bit overpriced in my...,Expansion - Hearts of Iron IV: Man the Guns,1,very good game although a bit overpriced in my...,EN,29,False
3,126.0,1086,676,False,Out of all the reviews I wrote This one is pro...,Dead by Daylight,1,out of all the reviews i wrote this one is pro...,EN,419,False
4,85.0,2139,612,False,Disclaimer I survivor main. I play games for f...,Dead by Daylight,1,disclaimer i survivor main. i play games for f...,EN,273,False


### 2.2 Rebuild group_id and split

In [11]:
from sklearn.model_selection import StratifiedGroupKFold

data['group_id'] = pd.factorize(data['review_clean'])[0]

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(
    sgkf.split(data['review_clean'], data['label'], groups=data['group_id'])
)

train = data.iloc[train_idx].reset_index(drop=True)
test  = data.iloc[test_idx].reset_index(drop=True)

print(f'Train: {len(train):,}')
print(f'Test:  {len(test):,}')

Train: 325,425
Test:  81,356


In [12]:
# The Notebook 04 test set had 81,356 rows. Same number -> same split.

assert len(test) == 81_356, f'Split mismatch: got {len(test):,}'
print('Split matches Notebook 04.')

Split matches Notebook 04.


## 3. Baseline Recap

### 3.1 Load artifacts

In [15]:
vect_baseline = joblib.load(os.path.join(BASELINE_DIR, 'sentiment_tfidf_vectorizer.joblib'))
logreg_baseline = joblib.load(os.path.join(BASELINE_DIR, 'logreg_sentiment.joblib'))

### 3.2 Predict on the restored test set

In [ ]:
X_test = vect_baseline.transform(test['review_clean'])

y_test = test['label']

In [22]:
y_pred_baseline = logreg_baseline.predict(X_test)
y_proba_baseline = logreg_baseline.predict_proba(X_test)[:, 1]

pr_auc_neg_baseline = average_precision_score(1 - y_test, 1 - y_proba_baseline) # negative class

print(f'PR-AUC (negative class): {pr_auc_neg_baseline:.3f}')
print(classification_report(y_test, y_pred_baseline, digits = 3))

PR-AUC (negative class): 0.894
              precision    recall  f1-score   support

           0      0.776     0.879     0.824     24860
           1      0.944     0.888     0.915     56496

    accuracy                          0.885     81356
   macro avg      0.860     0.884     0.870     81356
weighted avg      0.892     0.885     0.887     81356



## 4. Tokenization


### 4.1 Tokenizer

In [26]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

print('Vocab size:', tokenizer.vocab_size)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\adris\Desktop\Data_Science_Projects\steam_reviews_nlp\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adris\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Vocab size: 30522


In [32]:
sample_txt = test['review_clean'].iloc[10]

print('Sample Text:', sample_txt[:150])
print('Encoded Text', tokenizer(sample_txt))
print('Tokenized Text:', tokenizer.tokenize(sample_txt))

Sample Text: absolutely love it!!!!!
Encoded Text {'input_ids': [101, 7078, 2293, 2009, 999, 999, 999, 999, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Tokenized Text: ['absolutely', 'love', 'it', '!', '!', '!', '!', '!']


### 4.2 Vocab Coverage

In [30]:
for w in ["won't", "don't", "not good", "unplayable", "refund", "gamebreaking"]:
    print(f'{w!r:15} -> {tokenizer.tokenize(w)}')

"won't"         -> ['won', "'", 't']
"don't"         -> ['don', "'", 't']
'not good'      -> ['not', 'good']
'unplayable'    -> ['un', '##play', '##able']
'refund'        -> ['ref', '##und']
'gamebreaking'  -> ['game', '##break', '##ing']


The tokenizer splits almost everything:

| Input | WordPiece output |
|---|---|
| `won't` | `won`, `'`, `t` |
| `don't` | `don`, `'`, `t` |
| `unplayable` | `un`, `##play`, `##able` |
| `refund` | `ref`, `##und` |
| `gamebreaking` | `game`, `##break`, `##ing` |

In Notebook 04 this would have been a problem. TF-IDF treats every token as an
independent feature, so the `won` from `won't` lands in the same column as the
`won` from "I won the match". Order and context are gone. The custom
`token_pattern` was the only defense.

DistilBERT keeps the sequence. Self-attention reads `won`, `'`, `t` together,
so the vector for `won` in `won't recommend` differs from the vector for `won`
in `won the match`. Splitting the word costs nothing because the model puts the
pieces back together.

The `un` + `##play` + `##able` split is a gain, not a loss. The model sees the
negating prefix as a unit and can carry it over to words it has never
encountered. This is how a 30k vocabulary covers unbounded text, while our
TF-IDF baseline needed 259,426 features and still went blind on anything
outside them.

**Two open threads from 04 close here.** The custom `token_pattern` has no
counterpart in this notebook: the tokenizer ships with the pretrained model and
we do not touch it. Contraction expansion (`won't` to `will not`) would fix a
problem DistilBERT does not have.

**One consequence now.** Words expand into several tokens, `refund` into two.
Token counts run higher than word counts, so the `word_count` column from 02
cannot tell us what `MAX_LENGTH` to use. We have to tokenize and measure.

## 5. Sequence Length Analysis

### 5.1 Token Length Distribution

In [ ]:
def tokenize(batch):
    return tokenizer(batch['text'], padding = False, truncation = False)

### 5.2 Choose MAX_LENGTH

## 6. Data Preparation

## 7. Frozen Embedding as Features


## 8. Fine-Tuning DistilBERT

## 9. Evaluation on shared Test Set

## 10. Error Analysis

## 11. Cost vs. Benefit

## 12. Conclusions